# 1. Simple Tokenizer

In [25]:
import torch
from torch.utils.data import Dataset, DataLoader
import re
from typing import Dict, List
import tiktoken

In [19]:
# --- BƯỚC 1: CHUẨN BỊ DỮ LIỆU ---
sample_text = "Hello, world! This is a sample text. Hello world."

# 1. Tách từ bằng regex giống hệt trong Class SimpleTokenizer
# Mục đích: Đảm bảo cách tách lúc tạo vocab và lúc encode là giống nhau
preprocessed_tokens = re.split(r'([,.:;?_!"()\']|--|\s)', sample_text)

# 2. Lọc bỏ khoảng trắng thừa
preprocessed_tokens = [item.strip() for item in preprocessed_tokens if item.strip()]

# 3. Tạo tập hợp các từ unique
all_words = sorted(list(set(preprocessed_tokens)))

# 4. Thêm special tokens vào CUỐI
special_tokens = ["<|endoftext|>", "<|unk|>"]
all_words.extend(special_tokens)

# 5. Tạo Vocab
vocab = {w: i for i, w in enumerate(all_words)}

# Kiểm tra vocab
print("5 token cuối cùng trong vocab:", list(vocab.items())[-5:])
# Kết quả mong đợi: ..., ('world', ID), ('<|endoftext|>', ID), ('<|unk|>', ID)]


# --- BƯỚC 2: CLASS TOKENIZER ---
class SimpleTokenizer:
    def __init__(self, vocab: Dict[str, int]):
        self.str2int = vocab
        self.int2str = {i: s for s, i in vocab.items()}
    
    def encode(self, text: str) -> List[int]:
        # Tách từ
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        # Ánh xạ sang ID. Nếu từ không có trong vocab -> dùng ID của <|unk|>
        preprocessed = [
            item if item in self.str2int else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str2int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids: List[int]) -> str:
        text = " ".join([self.int2str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

# --- BƯỚC 3: CHẠY THỬ ---
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

tokenizer = SimpleTokenizer(vocab)

# Test 1: Câu nằm trong tập vocab (sẽ mã hóa đúng)
text1 = "Hello, world!" 
ids1 = tokenizer.encode(text1)
print(f"\nText 1: '{text1}'")
print(f"Encoded: {ids1}")
print(f"Decoded: {tokenizer.decode(ids1)}")

# Test 2: Câu có từ lạ (sẽ ra <|unk|>)
text2 = "Hello, Universe!" # Từ 'Universe' chưa có trong vocab
ids2 = tokenizer.encode(text2)
print(f"\nText 2: '{text2}'")
print(f"Encoded: {ids2}") 
print(f"Decoded: {tokenizer.decode(ids2)}")

5 token cuối cùng trong vocab: [('sample', 7), ('text', 8), ('world', 9), ('<|endoftext|>', 10), ('<|unk|>', 11)]
Vocab size: 12

Text 1: 'Hello, world!'
Encoded: [3, 1, 9, 0]
Decoded: Hello, world!

Text 2: 'Hello, Universe!'
Encoded: [3, 1, 11, 0]
Decoded: Hello, <|unk|>!


In [17]:
# Manual join 
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
special_tokens = " <|endoftext|> "
# how to concatenate two texts with special tokens in between
text_combined = special_tokens.join([text1, text2])
print(f"\nCombined Text: '{text_combined}'")


Combined Text: 'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'


# 2. byte pair encoding

In [24]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
            "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
            "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [ ]:
# max_length = 256
# len(token_ids) = 1000
# 1000 - 256 + 1 = 745
# 0 -> 745
# stride = 128

class LLMDataset:
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # tokenize the entire text
        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            terget_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(terget_chunk))
    
    # Returns the total number of rows in the dataset
    def __len__(self):
        return len(self.input_ids)
    